In [4]:
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
import os
from scipy.io import loadmat

In [5]:
! export HF_HOME='/scratch-shared/scur0412/hf_dir/'
! export HF_HUB_CACHE='/scratch-shared/scur0412/hf_dir/'

In [6]:
model_name = "Qwen/Qwen3-Embedding-8B"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(
    model_name, 
    torch_dtype="auto", 
    device_map="auto",
    trust_remote_code=True
)
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:18<00:00,  4.73s/it]


Qwen3Model(
  (embed_tokens): Embedding(151665, 4096)
  (layers): ModuleList(
    (0-35): 36 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
    )
  )
  (norm): Qwen

Data

In [7]:
exp2 = "/home/scur0412/NeuroNLP/data/participants/M02/data_243sentences.mat"
exp3 = "/home/scur0412/NeuroNLP/data/participants/M02/data_384sentences.mat"
data2 = loadmat(exp2, simplify_cells=True)
data3 = loadmat(exp3, simplify_cells=True)
sentences2 = data2['keySentences'].tolist()
sentences3 = data3['keySentences'].tolist()

/home/scur0412/NeuroNLP/neuro/lib64/python3.9/site-packages/scipy/io/matlab/_mio.py:227: MatReadWarning: Duplicate variable name "None" in stream - replacing previous with new
Consider mio5.varmats_from_mat to split file into single variable files
  matfile_dict = MR.get_variables(variable_names)


In [8]:
output_dir2 = "/home/scur0412/NeuroNLP/results/embeddings/Qwen_Embedder/Experiment2_243"
output_dir3 = "/home/scur0412/NeuroNLP/results/embeddings/Qwen_Embedder/Experiment3_384"
os.makedirs(output_dir2, exist_ok=True)
os.makedirs(output_dir3, exist_ok=True)

Extract embeddings

In [9]:
# Qwen3 has 37 layers 
num_layers = 37 

In [10]:
def extract_embeddings(text_input):
    inputs = tokenizer(text_input, padding=True, truncation=True, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    
    last_token_indexes = inputs['attention_mask'].sum(dim=1) - 1

    all_layer_embeddings = []

    for layer in outputs.hidden_states:
        batch_embed = layer[torch.arange(layer.size(0)), last_token_indexes]
        all_layer_embeddings.append(batch_embed.cpu().float().numpy())
        
    return np.array(all_layer_embeddings)

In [15]:
def all_experiments(sentences, output_dir, batch_size=32):
    layer_embeddings = {layer_idx: [] for layer_idx in range(num_layers)}

    print(f"\nProcessing {len(sentences)} sentences -> Saving to {output_dir}")

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        batch_embed = extract_embeddings(batch)
        for layer_index in range(num_layers):
            layer_embeddings[layer_index].append(batch_embed[layer_index])
        if (i // batch_size + 1) % 5 == 0: 
            print(f"  Completed batch {i//batch_size + 1}...")  

    for layer_index in range(num_layers):
        layer_data = np.concatenate(layer_embeddings[layer_index], axis=0)
        save_path = os.path.join(output_dir, f"qwen_layer{layer_index}.npy")  
        np.save(save_path, layer_data)

In [16]:
all_experiments(sentences2, output_dir2)


Processing 243 sentences -> Saving to /home/scur0412/NeuroNLP/results/embeddings/Qwen_Embedder/Experiment2_243
  Completed batch 5...


In [17]:
all_experiments(sentences3, output_dir3)


Processing 384 sentences -> Saving to /home/scur0412/NeuroNLP/results/embeddings/Qwen_Embedder/Experiment3_384
  Completed batch 5...
  Completed batch 10...
